In [ ]:
%%capture
import os, re

!pip install unsloth
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2
!pip install --upgrade transformers 

# Separando em Celulas.

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Hugging-Face-Token")

In [ ]:
from huggingface_hub import login

login(secret_value_0)

In [ ]:
import torch
from unsloth import FastVisionModel

MODEL_ID = "unsloth/gemma-4-E4B-it"

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=True,
)

print("modelo e tokenizer carregados com sucesso.")

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, # treinar os pesos de Visão Computacional
    finetune_language_layers=True, # treinar os pesos de linguagem
    finetune_attention_modules=True, # treinar os pesos de atenção
    finetune_mlp_modules=True,

    r=32, # indica o numero de parametros treinaveis nos adaptadores LoRA (r > -> capacidade maior, mas maior uso de memoria)
    lora_alpha=32, # escala a força do ajustamento do modelo (r == lora_alpha)
    lora_dropout=0, # previne o Overfitting.
    bias="none",
    random_state=3407, # numero fixo para garantir a reprodutibilidade dos resultados.
    use_rslora=False,
    loft_config=None
)

print("adicionando os adaptadores LoRA para treinamento.")

In [ ]:
system_think_instruction = """<|think|>
Siga essa linha de pensamento para a análise das imagens de Raio-x de Toráx (PA/Perfil).

1. Ossos e Partes Moles: Checar fraturas em costelas, clavículas e coluna; avaliar cúpulas diafragmáticas e buscar enfisema subcutâneo ou sombras mamárias.
2. Vias Aéreas: Avaliar se a traqueia está centralizada ou desviada e a perviedade dos brônquios principais.
3. Pulmões e Pleura: Buscar opacidades (nódulos, massas, consolidações) ou pneumotórax; checar se os seios costofrênicos estão livres ou velados (derrame).
4. Coração e Mediastino: Meça explicitamente o ICT. Avaliar se há cardiomegalia (ICT > 50%) e checar a anatomia do mediastino, botão aórtico e hilos.
5. Dispositivos (Se houver): Descrever presença e posicionamento de acessos, cateteres, tubos ou marca-passos.
6. Descrição das Alterações: Se houver achados anormais, descrever o tipo de lesão e a localização exata (ex: "opacidade em terço inferior do pulmão direito").
"""

system_report_format ="""
Você é um médico especialista no setor de Radiologia e seu objetivo é fornecer um laudo médico seguindo estritamente esse formato:

Título: 'RADIOGRAFIA DO TÓRAX – PA E PERFIL'
Findings: 
    - No formato de Bullet Points (-).
    - Qualquer aspecto anomalo que foi encontrado durante a análise das imagens deve ser indicado.
    - Caso não encontrar nenhuma anomlia na análise, não mencione.
    - Contudo, caso o aspecto das estruturas abaixo estejam preservadas, indique que está tudo bem de forma explicita:
        1. 'transparência pulmonar' (ex: - Transparência pulmonar preservada.)
        2. 'Seios costofrênicos' (ex: - Seios costofrênicos livres.)
        3. 'Mediastinos' (ex: Mediastino sem alterações.)
        4. 'Estruturas ósseas' (ex: Estruturas ósseas visualizadas sem alterações.)
"""

def convert_to_conversation(sample):
    conversation = [
        {
            "role": "system",
            "content": system_think_instruction
        },
        {
            "role": "system",
            "content": system_report_format
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["pa_image"]},
                {"type": "image", "image": sample["perfil_image"]},
                {"type": "text", "text": "Fornaça o Laudo das imagens anexadas (PA e Perfil) seguindo a linha de pensamento e o formato de laudo determinado."}
            ]
        },{
            "role": "assistant",
            "content": [
                {"type": "text", "text": sample["report"]}
            ]
        }
    ]

    return { "messages": conversation }

print("definindo o tipo de conversa a ser ajustado.")

In [ ]:
## cedule for loading the dataset.
from datasets import load_dataset

DATASET_ID = "guilhermenf/huac_chest_xray_reports_images"

dataset_train = load_dataset(DATASET_ID, split="train")
dataset_test = load_dataset(DATASET_ID, split="test")

print("carregando os datasets.")

In [ ]:
# turning the dataset to conversation dataset
dataset_conversation_train = [convert_to_conversation(sample) for sample in dataset_train]
dataset_conversation_test = [convert_to_conversation(sample) for sample in dataset_test]

print("convertendo o dataset para o formato correto de treinamento.")

In [ ]:
### cedule for the fine-tuning trainament.
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

training_configs = SFTConfig(
    # --- Parâmetros de Volume e Performance ---
    per_device_train_batch_size = 4,           # Tamanho do lote por GPU. 2 é seguro para evitar OOM em 8GB.
    gradient_accumulation_steps = 4,           # Acumula gradientes para simular um batch real de 8 (2 * 4).
    max_steps = 150,                          # Alinhado com a "Zona Mágica" de 2k-3k passos do Reddit.
    warmup_steps = 10,                        # 5% do total de passos para estabilizar o início do treino.
    learning_rate = 2e-4,                      # Valor padrão sólido para LoRA/QLoRA em LLMs.
    ddp_find_unused_parameters = False,
    
    # --- Otimização de Memória e Estabilidade ---
    optim = "paged_adamw_8bit",                # Versão "paged" é mais resiliente a picos de memória que a "8bit" comum.
    weight_decay = 0.01,                       # Regularização para evitar que o modelo decore o dataset (overfitting).
    lr_scheduler_type = "cosine",              # "Cosine" costuma ser superior ao "linear" para treinos longos.
    
    # --- Monitoramento e Logs ---
    logging_steps = 1,                        # Logar a cada 10 passos para não poluir o terminal, mas manter o controle.
    eval_strategy = "steps",                   # Avaliar por passos (essencial se max_steps for usado).
    eval_steps = 50,                          # Avaliar a cada 500 passos para encontrar o "sweet spot".
    save_steps = 50,                          # Salva checkpoints. Lembra da dica de testar épocas diferentes?
    
    # --- Configurações de Infraestrutura ---
    seed = 3407,                               # Seed famosa na comunidade por ter (anedoticamente) melhor performance.
    report_to = "none",                        # Mude para "wandb" ou "tensorboard" se quiser gráficos de perda.

    # --- Configurações Obrigatórias para Vision/Unsloth ---
    remove_unused_columns = False,
    dataset_text_field = "messages",
    dataset_kwargs = {"skip_prepare_dataset": True},
    max_length = 2048,

    # --- Configurações de alocamento dos resultados do LoRA ---
    push_to_hub = True,                # Ativa o upload automático
    hub_model_id = "joaoneto9/gemma4-x-ray-reports-LoRA-adapters", # O nome que aparecerá no HF
    hub_strategy = "end",              # Sobe tudo apenas ao final do treino
    hub_token = secret_value_0,      # Opcional se já fez login via CLI
    output_dir = "outputs",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Ele faz a mágica
    train_dataset = dataset_conversation_train,
    eval_dataset = dataset_conversation_test,
    args = training_configs,
)

print("configurando os parametros de treinamento do modelo.")

In [ ]:
stats = uns

In [ ]:
### cedule looking the inference of the model before the fine-tuning.
FastVisionModel.for_inference(model) # Enable for inference!

conversation = [
        {
            "role": "system",
            "content": system_think_instruction
        },
        {
            "role": "system",
            "content": system_report_format
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Fornaça o Laudo das imagens abaixo (PA e Perfil) seguindo a linha de pensamento e o formato de laudo determinado."},
                {"type": "image"},
                {"type": "image"}
            ]
        }
    ]

input_text = tokenizer.apply_chat_template(conversation, add_generation_prompt = True)
inputs = tokenizer(
    [dataset_test[0]["pa_image"], dataset_test[0]["perfil_image"]],
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)